# Fusion Evaluation — Pair Comparison

In [1]:
import sys
sys.path.insert(0, r"C:\FYP\src")
from utils.config import (
    CLINICAL_MODEL_COMPARISON_PATH, IMAGING_MODEL_COMPARISON_PATH,
    CLINICAL_OOF_PREDICTIONS_PATH, IMAGING_OOF_PREDICTIONS_PATH,
    CHECKPOINTS_CLINICAL_FINAL_DIR, CHECKPOINTS_IMAGING_FINAL_DIR,
    FUSION_PAIR_COMPARISON_PATH, EVAL_FUSION_DIR, RESULTS_FUSION_DIR, ensure_dirs,
)

import json
import textwrap

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

ensure_dirs()
pd.set_option("display.width", 120)
print("Imports OK")

Imports OK


## Load Each Branch's Own Results (no recomputation)

In [2]:
WINNER_IMAGING = "resnet50_unet"
WINNER_TABULAR = "XGBoost"
IMAGING_SCHEME = "5-Fold CV"          # imaging's only evaluation scheme
TABULAR_SCHEME = "Repeated 5x20 CV"   # clinical's most robust scheme (100 folds); same one clinical_final_fit.ipynb references

clinical_comparison_df = pd.read_csv(CLINICAL_MODEL_COMPARISON_PATH)
imaging_comparison_df = pd.read_csv(IMAGING_MODEL_COMPARISON_PATH)

tabular_row = clinical_comparison_df[
    (clinical_comparison_df["scheme"] == TABULAR_SCHEME) & (clinical_comparison_df["model"] == WINNER_TABULAR)
].iloc[0]
imaging_row = imaging_comparison_df[
    (imaging_comparison_df["scheme"] == IMAGING_SCHEME) & (imaging_comparison_df["model"] == WINNER_IMAGING)
].iloc[0]

with open(CHECKPOINTS_IMAGING_FINAL_DIR / "model_card.json") as f:
    imaging_model_card = json.load(f)

IMAGING_CAVEAT = (
    "KNOWN LIMITATION (checkpoints/imaging/final/model_card.json, confound-check investigation): "
    + imaging_model_card["known_limitations"]["confound_check_summary"]
)

print(f"Tabular winner: {WINNER_TABULAR}  ({TABULAR_SCHEME})")
print(tabular_row[["precision_mean", "recall_mean", "specificity_mean", "f1_mean", "auc_mean", "tn", "fp", "fn", "tp"]].to_string())
print(f"\nImaging winner: {WINNER_IMAGING}  ({IMAGING_SCHEME})")
print(imaging_row[["precision_mean", "recall_mean", "specificity_mean", "f1_mean", "roc_auc_mean", "tn", "fp", "fn", "tp"]].to_string())

print("\n" + "=" * 100)
print("IMAGING CAVEAT (attached to every imaging metric in this notebook):")
print(textwrap.fill(IMAGING_CAVEAT, 100))
print("=" * 100)

Tabular winner: XGBoost  (Repeated 5x20 CV)
precision_mean       0.776
recall_mean         0.7433
specificity_mean    0.8884
f1_mean             0.7573
auc_mean            0.9077
tn                    6947
fp                     873
fn                    1022
tp                    2958

Imaging winner: resnet50_unet  (5-Fold CV)
precision_mean      0.991981
recall_mean         0.962841
specificity_mean    0.969637
f1_mean             0.976785
roc_auc_mean        0.993425
tn                     18031
fp                       585
fn                      2679
tp                     69398

IMAGING CAVEAT (attached to every imaging metric in this notebook):
KNOWN LIMITATION (checkpoints/imaging/final/model_card.json, confound-check investigation): A
rigorous, 6-round confound-check investigation (Grad-CAM + 5 other attribution methods, then causal
occlusion testing) found this architecture's detection head originally relied MORE on the BOX=320
packing's synthetic padding than on the real tu

## The One Real Pair: `resnet50_unet` vs. `XGBoost`

In [3]:
pair_row = {
    "imaging_model": WINNER_IMAGING,
    "imaging_scheme": IMAGING_SCHEME,
    "imaging_precision": float(imaging_row["precision_mean"]),
    "imaging_recall": float(imaging_row["recall_mean"]),
    "imaging_specificity": float(imaging_row["specificity_mean"]),
    "imaging_f1": float(imaging_row["f1_mean"]),
    "imaging_roc_auc": float(imaging_row["roc_auc_mean"]),
    "imaging_confusion_matrix_tn_fp_fn_tp": f"{int(imaging_row.tn)}/{int(imaging_row.fp)}/{int(imaging_row.fn)}/{int(imaging_row.tp)}",
    "imaging_known_limitations_caveat": IMAGING_CAVEAT,
    "tabular_model": WINNER_TABULAR,
    "tabular_scheme": TABULAR_SCHEME,
    "tabular_precision": float(tabular_row["precision_mean"]),
    "tabular_recall": float(tabular_row["recall_mean"]),
    "tabular_specificity": float(tabular_row["specificity_mean"]),
    "tabular_f1": float(tabular_row["f1_mean"]),
    "tabular_roc_auc": float(tabular_row["auc_mean"]),
    "tabular_confusion_matrix_tn_fp_fn_tp": f"{int(tabular_row.tn)}/{int(tabular_row.fp)}/{int(tabular_row.fn)}/{int(tabular_row.tp)}",
    "note": (
        "No joint/paired metric exists across these two rows -- each branch's metrics are computed "
        "independently, on its own patients, from its own model_comparison.csv. No patient in this "
        "project has both a CT scan and a urine sample, so there is no ground truth for a fused "
        "prediction. See this notebook's markdown and final summary cell."
    ),
}
pair_comparison_df = pd.DataFrame([pair_row])
pair_comparison_df.to_csv(FUSION_PAIR_COMPARISON_PATH, index=False)
print(f"Saved {FUSION_PAIR_COMPARISON_PATH}")
pair_comparison_df.T

Saved C:\FYP\results\fusion\pair_comparison.csv


,0
imaging_model,resnet50_unet
imaging_scheme,5-Fold CV
imaging_precision,0.991981
imaging_recall,0.962841
imaging_specificity,0.969637
imaging_f1,0.976785
imaging_roc_auc,0.993425
imaging_confusion_matrix_tn_fp_fn_tp,18031/585/2679/69398
imaging_known_limitations_caveat,KNOWN LIMITATION (checkpoints/imaging/final/mo...
tabular_model,XGBoost


In [4]:
metric_labels = ["precision", "recall", "specificity", "f1", "roc_auc"]
imaging_vals = [pair_row[f"imaging_{m}"] for m in metric_labels]
tabular_vals = [pair_row[f"tabular_{m}"] for m in metric_labels]

x = np.arange(len(metric_labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.bar(x - width / 2, imaging_vals, width, label=f"Imaging: {WINNER_IMAGING}")
ax.bar(x + width / 2, tabular_vals, width, label=f"Tabular: {WINNER_TABULAR}")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title("Each branch's own metrics, on its own patients/scheme -- NOT a joint or fused benchmark")
ax.legend()

fig.text(
    0.5, -0.02,
    "\n".join(textwrap.wrap(IMAGING_CAVEAT, 118)),
    ha="center", va="top", fontsize=7.5, family="monospace",
)

fig_path = EVAL_FUSION_DIR / "pair_comparison.png"
fig.tight_layout()
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved {fig_path}")

Saved C:\FYP\outputs\eval\fusion\pair_comparison.png


## Calibration Quality — Brier Score + Reliability Curve, From Each Branch's Own OOF File

In [5]:
clinical_oof_df = pd.read_csv(CLINICAL_OOF_PREDICTIONS_PATH)
imaging_oof_df = pd.read_csv(IMAGING_OOF_PREDICTIONS_PATH)

tabular_oof = clinical_oof_df[clinical_oof_df["model"] == WINNER_TABULAR].reset_index(drop=True)
imaging_oof = imaging_oof_df[imaging_oof_df["candidate"] == WINNER_IMAGING].reset_index(drop=True)

tabular_calibrator = joblib.load(CHECKPOINTS_CLINICAL_FINAL_DIR / "calibrator.pkl")
imaging_calibrator = joblib.load(CHECKPOINTS_IMAGING_FINAL_DIR / "calibrator.pkl")

tabular_calibrated_proba = tabular_calibrator.predict_proba(tabular_oof[["y_proba"]].to_numpy())[:, 1]
imaging_calibrated_proba = imaging_calibrator.predict_proba(imaging_oof[["y_proba"]].to_numpy())[:, 1]

brier_results = {
    "tabular_raw": brier_score_loss(tabular_oof["y_true"], tabular_oof["y_proba"]),
    "tabular_calibrated": brier_score_loss(tabular_oof["y_true"], tabular_calibrated_proba),
    "imaging_raw": brier_score_loss(imaging_oof["y_true"], imaging_oof["y_proba"]),
    "imaging_calibrated": brier_score_loss(imaging_oof["y_true"], imaging_calibrated_proba),
}

print(f"Tabular OOF rows: {len(tabular_oof):,} ({tabular_oof['sample_id'].nunique()} unique patients)")
print(f"Imaging OOF rows: {len(imaging_oof):,} ({imaging_oof['patient_id'].nunique()} unique patients, slice-level)")
print("\nBrier score (0 = perfect; lower is better; ~0.25 = a coin flip on a balanced set):")
for k, v in brier_results.items():
    print(f"  {k:<20} {v:.4f}")
print(
    "\nCalibrated numbers are mildly optimistic (calibrator fit on this same OOF set -- see markdown "
    "above); shown to confirm the calibrator improves calibration, not as an independent estimate."
)
print("\nIMAGING CAVEAT applies to every imaging number above:")
print(textwrap.fill(IMAGING_CAVEAT, 100))

Tabular OOF rows: 11,800 (590 unique patients)
Imaging OOF rows: 90,693 (361 unique patients, slice-level)

Brier score (0 = perfect; lower is better; ~0.25 = a coin flip on a balanced set):
  tabular_raw          0.1213
  tabular_calibrated   0.1189
  imaging_raw          0.0303
  imaging_calibrated   0.0282

Calibrated numbers are mildly optimistic (calibrator fit on this same OOF set -- see markdown above); shown to confirm the calibrator improves calibration, not as an independent estimate.

IMAGING CAVEAT applies to every imaging number above:
KNOWN LIMITATION (checkpoints/imaging/final/model_card.json, confound-check investigation): A
rigorous, 6-round confound-check investigation (Grad-CAM + 5 other attribution methods, then causal
occlusion testing) found this architecture's detection head originally relied MORE on the BOX=320
packing's synthetic padding than on the real tumour region (Round 4: 2.03x a random-patch control,
p=0.044). A random-resized-crop augmentation (augment=

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))

panels = [
    (axes[0], "tabular", f"Tabular: {WINNER_TABULAR}", tabular_oof, tabular_calibrated_proba),
    (axes[1], "imaging", f"Imaging: {WINNER_IMAGING}", imaging_oof, imaging_calibrated_proba),
]
for ax, prefix, title, oof, calib_proba in panels:
    frac_pos_raw, mean_pred_raw = calibration_curve(oof["y_true"], oof["y_proba"], n_bins=10)
    frac_pos_cal, mean_pred_cal = calibration_curve(oof["y_true"], calib_proba, n_bins=10)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect calibration")
    ax.plot(mean_pred_raw, frac_pos_raw, marker="o", label=f"Raw (Brier={brier_results[prefix + '_raw']:.4f})")
    ax.plot(mean_pred_cal, frac_pos_cal, marker="s", label=f"Calibrated (Brier={brier_results[prefix + '_calibrated']:.4f})")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.suptitle("Reliability diagrams -- each branch's own OOF predictions (not a joint calibration)")
fig.text(
    0.5, -0.06,
    "\n".join(textwrap.wrap("IMAGING CAVEAT: " + imaging_model_card["known_limitations"]["confound_check_summary"], 130)),
    ha="center", va="top", fontsize=7, family="monospace",
)

fig_path2 = EVAL_FUSION_DIR / "calibration_curves.png"
fig.tight_layout()
fig.savefig(fig_path2, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved {fig_path2}")

Saved C:\FYP\outputs\eval\fusion\calibration_curves.png


## Worked Examples: Illustrative Fused Predictions (NOT a benchmark, NOT real paired patients)

In [ ]:
N_EXAMPLES = 5


def pick_spread(sorted_arr, n):
    idx = np.linspace(0, len(sorted_arr) - 1, n).astype(int)
    return sorted_arr[idx]


tabular_raw_sorted = np.sort(tabular_oof["y_proba"].to_numpy())
imaging_raw_sorted = np.sort(imaging_oof["y_proba"].to_numpy())

tabular_spread_raw = pick_spread(tabular_raw_sorted, N_EXAMPLES)
imaging_spread_raw = pick_spread(imaging_raw_sorted, N_EXAMPLES)

tabular_spread_calibrated = tabular_calibrator.predict_proba(tabular_spread_raw.reshape(-1, 1))[:, 1]
imaging_spread_calibrated = imaging_calibrator.predict_proba(imaging_spread_raw.reshape(-1, 1))[:, 1]

rng = np.random.default_rng(0)
imaging_spread_calibrated_shuffled = rng.permutation(imaging_spread_calibrated)

worked_examples_df = pd.DataFrame({
    "hypothetical_patient": [f"H{i + 1}" for i in range(N_EXAMPLES)],
    "imaging_calibrated_proba": imaging_spread_calibrated_shuffled,
    "tabular_calibrated_proba": tabular_spread_calibrated,
})
worked_examples_df["fused_score_simple_mean"] = worked_examples_df[
    ["imaging_calibrated_proba", "tabular_calibrated_proba"]
].mean(axis=1)
worked_examples_df["illustrative_risk_band"] = pd.cut(
    worked_examples_df["fused_score_simple_mean"], bins=[-0.01, 0.33, 0.66, 1.0], labels=["low", "medium", "high"]
)

worked_examples_path = RESULTS_FUSION_DIR / "worked_fusion_examples.csv"
worked_examples_df.to_csv(worked_examples_path, index=False)

print("ILLUSTRATIVE ONLY -- fabricated pairings of two real, independently-calibrated scores.")
print("NOT a benchmark, NOT real patients, NO accuracy number (no ground truth exists -- rule 1).\n")
print(worked_examples_df.to_string(index=False))
print(f"\nSaved {worked_examples_path}")
print("\nIMAGING CAVEAT applies to every imaging_calibrated_proba value above:")
print(textwrap.fill(IMAGING_CAVEAT, 100))

ILLUSTRATIVE ONLY -- fabricated pairings of two real, independently-calibrated scores.
NOT a benchmark, NOT real patients, NO accuracy number (no ground truth exists -- rule 1).

hypothetical_patient  imaging_calibrated_proba  tabular_calibrated_proba  fused_score_simple_mean illustrative_risk_band
                  H1                  0.994695                  0.082159                 0.538427                 medium
                  H2                  0.994695                  0.083512                 0.539104                 medium
                  H3                  0.994695                  0.108936                 0.551815                 medium
                  H4                  0.090900                  0.712764                 0.401832                 medium
                  H5                  0.988372                  0.883994                 0.936183                   high

Saved C:\FYP\results\fusion\worked_fusion_examples.csv

IMAGING CAVEAT applies to every imagin

## Summary

In [8]:
print("=== SUMMARY ===\n")
print(f"Pair compared: {WINNER_IMAGING} (imaging) vs. {WINNER_TABULAR} (tabular)\n")
print(f"{'metric':<14}{'imaging':>12}{'tabular':>12}")
for m, imaging_col, tabular_col in [
    ("precision", "imaging_precision", "tabular_precision"),
    ("recall", "imaging_recall", "tabular_recall"),
    ("specificity", "imaging_specificity", "tabular_specificity"),
    ("f1", "imaging_f1", "tabular_f1"),
    ("roc_auc", "imaging_roc_auc", "tabular_roc_auc"),
]:
    print(f"{m:<14}{pair_row[imaging_col]:>12.4f}{pair_row[tabular_col]:>12.4f}")

print(
    f"\nBrier score -- calibrated: imaging={brier_results['imaging_calibrated']:.4f}, "
    f"tabular={brier_results['tabular_calibrated']:.4f} (lower is better)"
)

print(
    "\nNO JOINT METRIC: the numbers above are two SEPARATE evaluations, on two SEPARATE, unpaired "
    "cohorts (MSD/NIH CT scans vs. the Debernardi et al. urine cohort). No patient has both, so "
    "there is no way to score a combined prediction -- this notebook does not compute or report a "
    "joint precision/recall/F1/ROC-AUC/confusion matrix, per rule 1. The worked-example fusion rows "
    "above are illustrative only, not a benchmark, and used fabricated (shuffled) pairings.\n"
)

print("IMAGING CAVEAT -- attached everywhere imaging's metrics appear in this notebook")
print("(the pair table row/column, the pair_comparison.png annotation, the calibration print-out,")
print("the calibration_curves.png annotation, and every worked-example row's imaging score):\n")
print(textwrap.fill(IMAGING_CAVEAT, 100))

=== SUMMARY ===

Pair compared: resnet50_unet (imaging) vs. XGBoost (tabular)

metric             imaging     tabular
precision           0.9920      0.7760
recall              0.9628      0.7433
specificity         0.9696      0.8884
f1                  0.9768      0.7573
roc_auc             0.9934      0.9077

Brier score -- calibrated: imaging=0.0282, tabular=0.1189 (lower is better)

NO JOINT METRIC: the numbers above are two SEPARATE evaluations, on two SEPARATE, unpaired cohorts (MSD/NIH CT scans vs. the Debernardi et al. urine cohort). No patient has both, so there is no way to score a combined prediction -- this notebook does not compute or report a joint precision/recall/F1/ROC-AUC/confusion matrix, per rule 1. The worked-example fusion rows above are illustrative only, not a benchmark, and used fabricated (shuffled) pairings.

IMAGING CAVEAT -- attached everywhere imaging's metrics appear in this notebook
(the pair table row/column, the pair_comparison.png annotation, the cal